# 프롬프트 템플릿(PromptTemplate) 실습

LangChain의 `PromptTemplate`과 `ChatPromptTemplate`을 이용해 프롬프트를 템플릿화하고, `partial_variables`로 값을 고정하거나 동적으로 채우는 방법, YAML 파일로 프롬프트를 관리하는 방법을 실습한다.

In [1]:
# ! pip --version

In [2]:
# !pip install dotenv

In [3]:
# !pip install -U langchain langchain_openai

In [4]:
# 단축키: [ctrl + shift + p] >>> "Python: 언어 서버 다시 시작 (Python: Restart Language Server)"
from dotenv import load_dotenv

load_dotenv()


True

In [5]:
from langchain_openai import ChatOpenAI 

# 기본 모델(gpt-3.5-turbo)로 ChatOpenAI 인스턴스 생성
llm = ChatOpenAI()

## 1. PromptTemplate 기본 사용법

`PromptTemplate.from_template()`으로 문자열 템플릿에서 바로 프롬프트를 만들 수 있다. `{country}`처럼 중괄호로 감싼 부분이 나중에 채워질 변수(`input_variables`)가 된다.

In [6]:
from langchain_core.prompts import PromptTemplate

template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [7]:
prompt = prompt.format(country="대한민국")
prompt

'대한민국의 수도는 어디인가요?'

In [8]:
template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate.from_template(template)

chain = prompt | llm

In [9]:
chain.invoke("대한민국").content 

'대한민국의 수도는 서울입니다.'

## 2. PromptTemplate 직접 생성하기

`from_template` 대신 `PromptTemplate(template=..., input_variables=[...])`처럼 생성자를 직접 호출해서 만들 수도 있다. 결과는 동일하다.

In [10]:
template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate(
    template = template,
    input_variables=["country"]
)

prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [11]:
prompt.format(country="대한민국")

'대한민국의 수도는 어디인가요?'

## 3. partial_variables로 값 고정하기

템플릿에 변수가 여러 개일 때 그중 일부를 `partial_variables`로 미리 고정해두면 나머지 변수만 채워서 사용할 수 있다. 이미 만든 프롬프트에 `.partial()`을 호출해서 나중에 값을 고정하는 것도 가능하다.

참고로 아래 코드에는 `input_variables=["countnry1"]`처럼 오타가 있는데, `PromptTemplate`은 실제 템플릿 문자열(`{country1}`)을 기준으로 `input_variables`를 다시 계산하기 때문에 결과에는 오타 없이 `country1`로 정상 표시된다.

In [12]:
template = "{country1}과 {country2}의 수도는 각각 어디인가요?" 

prompt = PromptTemplate(
    template=template,
    input_variables=["countnry1"],  # 오타(countnry1)지만 아래 결과처럼 템플릿 문자열 기준으로 재계산된다
    partial_variables={
        "country2": "미국"  # country2는 항상 '미국'으로 고정
    },
)

prompt

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '미국'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [13]:
prompt.format(country1="대한민국")

'대한민국과 미국의 수도는 각각 어디인가요?'

In [14]:
prompt_partial = prompt.partial(country2="캐나다")
prompt_partial 

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '캐나다'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [15]:
prompt_partial.format(country1="대한민국")

'대한민국과 캐나다의 수도는 각각 어디인가요?'

In [16]:
chain = prompt_partial | llm 

In [17]:
chain.invoke("대한민국").content

'대한민국의 수도는 서울이고, 캐나다의 수도는 오타와입니다.'

In [18]:
chain.invoke({"country1":"대한민국", "country2": "호주"}).content

'대한민국의 수도는 서울이고, 호주의 수도는 캔버라입니다.'

## 4. partial_variables에 함수 넣어 동적 값 만들기

`partial_variables`에는 고정된 값뿐 아니라 **호출 가능한 함수**도 넣을 수 있다. 프롬프트를 `format()`할 때마다 함수가 실행되어 값이 채워지므로, '오늘 날짜'처럼 매번 달라지는 값을 넣을 때 유용하다.

In [19]:
from datetime import datetime 

datetime.now().strftime("%B %d")

'September 15'

In [20]:
def get_today():
    return datetime.now().strftime("%B %d")

In [21]:
prompt = PromptTemplate(
    template="오늘의 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 생년월일을 표기해주세요.",
    input_variables=["n"],
    partial_variables={
        "today": get_today  # 함수를 넣으면 format() 호출 시점마다 실행되어 오늘 날짜로 채워진다
    }
)

In [22]:
prompt.format(n=3)

'오늘의 날짜는 September 15입니다. 오늘이 생일인 유명인 3명을 나열해 주세요. 생년월일을 표기해주세요.'

In [23]:
chain = prompt | llm

In [24]:
print(chain.invoke(3).content)

1. Prince Harry - 1984년 9월 15일
2. Tommy Lee Jones - 1946년 9월 15일
3. Agatha Christie - 1890년 9월 15일


In [25]:
print(chain.invoke({"today":"Jan 02","n":3}).content)

1. 김태희 - 1980년 3월 29일
2. 레드벨벳 아이린 - 1991년 3월 29일
3. 제이슨 스테이섬 - 1967년 1월 2일


## 5. YAML 파일로 프롬프트 관리하기

프롬프트가 길어지면 코드에서 문자열로 관리하기보다 `.yaml` 파일로 분리해서 `load_prompt()`로 불러오는 게 편하다. `prompts/fruit_color.yaml`, `prompts/capital.yaml` 두 개를 만들어서 실습했다.

`capital.yaml`:
```yaml
_type: "prompt"
template: |
  {country}의 수도에 대해서 알려주세요.
  수도의 특징을 다음의 양식에 맞게 정리해 주세요.
  300자 내외로 작성해 주세요.
  한글로 작성해 주세요.
  ----
  [양식]
  1. 면적
  2. 인구
  3. 역사적 장소
  4. 특산품

  #Answer:
input_variables: ["country"]
```

In [26]:
from langchain_core.prompts import load_prompt 

# yaml 파일에 저장해둔 프롬프트를 그대로 불러오기
prompt = load_prompt("prompts/fruit_color.yaml", encoding="utf-8")
prompt

C:\Users\user\AppData\Local\Temp\ipykernel_11184\3315787428.py:4: LangChainDeprecationWarning: The function `load_prompt` was deprecated in LangChain 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  prompt = load_prompt("prompts/fruit_color.yaml", encoding="utf-8")


PromptTemplate(input_variables=['fruit'], input_types={}, partial_variables={}, template='{fruit}의 색깔이 뭐야?')

In [27]:
prompt.format(fruit="사과")

'사과의 색깔이 뭐야?'

In [28]:
prompt2 = load_prompt("prompts/capital.yaml", encoding="utf-8")
print(prompt2.format(country="대한민국"))

대한민국의 수도에 대해서 알려주세요.
수도의 특징을 다음의 양식에 맞게 정리해 주세요.
300자 내외로 작성해 주세요.
한글로 작성해 주세요.
----
[양식]
1. 면적
2. 인구
3. 역사적 장소
4. 특산품

#Answer:



## 6. ChatPromptTemplate 기본 사용법

`PromptTemplate`은 단순 문자열 프롬프트고, `ChatPromptTemplate`은 채팅 모델에 맞게 여러 개의 메시지(system/human/ai)를 하나의 템플릿으로 묶어서 관리할 수 있게 해준다. `from_template()`을 쓰면 human 메시지 하나짜리 템플릿을 간단히 만들 수 있다.

In [29]:
from langchain_core.prompts import ChatPromptTemplate 

chat_prompt = ChatPromptTemplate.from_template("{country}의 수도는 어디인가요?")
chat_prompt

ChatPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?'), additional_kwargs={})])

In [30]:
chat_prompt.format(country="대한민국")

'Human: 대한민국의 수도는 어디인가요?'

## 7. from_messages로 여러 역할의 메시지 구성하기

`from_messages()`에 `(역할, 내용)` 튜플 리스트를 넘기면 system/human/ai 메시지를 순서대로 구성할 수 있다. `format_messages()`로 변수를 채우면 실제 `SystemMessage`, `HumanMessage`, `AIMessage` 객체 리스트가 만들어지고, 이를 그대로 `llm.invoke()`에 넣어 호출할 수 있다.

In [31]:
from langchain_core.prompts import ChatPromptTemplate 

# (역할, 메시지) 튜플 리스트로 system/human/ai 대화 흐름을 템플릿으로 구성
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name}입니다."),
        ("human", "반가워요!"),
        ("ai", "안녕하세요! 무엇을 도와드릴까요?"),
        ("human", "{user_input}"),
    ]
)

# 변수를 채워서 실제 메시지 객체 리스트로 변환
messages = chat_template.format_messages(
    name="알파고", user_input="당신의 이름은 무엇입니까?"
)
messages 

[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 알파고입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='반가워요!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='당신의 이름은 무엇입니까?', additional_kwargs={}, response_metadata={})]

In [32]:
llm.invoke(messages).content

'제이름은 알파고입니다. 어떻게 도와드릴까요?'

In [33]:
chain = chat_template | llm 

In [34]:
chain.invoke({"name": "아이쇼스피드", "user_input": "당신의 이름은 무엇입니까?"}).content

'제 이름은 아이쇼스피드입니다. 부르실 때는 아이쇼스피드라고 말해주세요. 어떻게 도와드릴까요?'

## 8. MessagesPlaceholder로 대화 이력 넣기

`MessagesPlaceholder(variable_name="conversation")`을 템플릿 중간에 끼워 넣으면, 실행 시점에 이전 대화 메시지 리스트를 그 자리에 통째로 삽입할 수 있다. 대화 요약처럼 '지금까지의 대화'를 프롬프트에 포함시켜야 할 때 유용하다. 마지막엔 `StrOutputParser()`를 체인에 붙여서 결과를 바로 문자열로 받는다.

In [35]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다ㅣ.",
        ),
        MessagesPlaceholder(variable_name="conversation"),  # 여기에 이전 대화 메시지들이 통째로 삽입된다
        ("human", "지금까지의 대화를 {word_count} 단어로 요약합니다."),
    ]
)
chat_prompt

ChatPromptTemplate(input_variables=['conversation', 'word_count'], input_types={'conversation': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annota

In [36]:
formatted_chat_prompt = chat_prompt.format(
    word_count=5,
    conversation=[
        ("human", "안녕하세요! 저는 오늘 새로 입사한 아이쇼스피드입니다. 만나서 반갑습니다."),
        ("ai", "반가워요! 앞으로 잘 부탁드립니다.")
    ]
)

print(formatted_chat_prompt)

System: 당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다ㅣ.
Human: 안녕하세요! 저는 오늘 새로 입사한 아이쇼스피드입니다. 만나서 반갑습니다.
AI: 반가워요! 앞으로 잘 부탁드립니다.
Human: 지금까지의 대화를 5 단어로 요약합니다.


In [37]:
chain = chat_prompt | llm | StrOutputParser()

In [38]:
chain.invoke(
    {
        "word_count": 5,
        "conversation": [
        ("human", "안녕하세요! 저는 오늘 새로 입사한 아이쇼스피드입니다. 만나서 반갑습니다."),
        ("ai", "반가워요! 앞으로 잘 부탁드립니다.")
        ],
    }
)

'인사와 소속 소개했습니다.'